# Cryptography - Take Home Exam

## Problem 1

Consider the safeprime

In [1]:
p = 34812256807200112032747336042516347072852568000061819419497014720603142778393164883078864092540099747

How many integers $g$ with $1 \le g \le 100$ are primitive roots modulo $p$?  Output them as a list.

In [3]:
primitive_roots = []

q = (p - 1) // 2

for g in range(1, 101):
    if pow(g, 2, p) != 1 and pow(g, q, p) != 1:
        primitive_roots.append(g)

print(primitive_roots)

[2, 5, 6, 7, 8, 15, 18, 20, 21, 22, 23, 24, 26, 28, 31, 32, 34, 38, 43, 45, 50, 54, 55, 58, 60, 63, 65, 66, 69, 70, 71, 72, 73, 74, 77, 78, 80, 82, 83, 84, 85, 88, 91, 92, 93, 94, 95, 96, 97, 98]


## Problem 2

This problem concerns the Diffie-Hellman key exchange.
1. Generate a large probable safeprime $p$ with $100$ decimal digits.  Use your own code for the strong pseudoprime test to do this.  Using $5$ random bases will suffice.

In [4]:
def pseudo_test(n, b):
    if gcd(b, n) != 1:
        return False
    
    if pow(b, n - 1, n) != 1:
        return False
    
    return True

In [5]:
def strong_pseudo_test(n,b):
    if not pseudo_test(n,b):
        return False

    q = n - 1
    k = 0
    while q % 2 == 0:
        q = q // 2
        k += 1

    if pow(b, q, n) == 1:
        return True

    for i in range(k):
        if pow(b, (2**i) * q, n) == n-1:
            return True
    
    return False

In [6]:
def miller_rabin(n, k):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False

    for i in range(k):
        b = randint(2, n - 2)
        if not strong_pseudo_test(n, b):
            return False

    return True

In [7]:
def sbsgs_dl(g, p, h):
    n = 1 + floor(sqrt(p - 1))
    g = Mod(g, p)
    h = Mod(h, p)
    
    baby_steps = []
    for i in range(n):
        baby_steps.append(g^i)

    giant_steps = []
    for j in range(n):
        giant_steps.append(h * g^(-j*n))

    for i in range(n):
        for j in range(n):
            if baby_steps[i] == giant_steps[j]:
                y = i + j*n
                return y

In [8]:
while True:
    p = randint(10^99, 10^100 - 1)

    if p % 2 == 0:
        p += 1

    q = (p - 1) // 2

    if miller_rabin(p, 5) and miller_rabin(q, 5):
        print("p =", p)
        print("q =", q)
        break

p = 1811358398361379287906878500003529531316749656723863095122953803596584355652194899082961044265414603
q = 905679199180689643953439250001764765658374828361931547561476901798292177826097449541480522132707301


2.  Suppose you and myself are generating a shared secret using Diffie-Hellman using the safeprime $p$ from the previous part.  Suppose we agree on $g = 65535$.  Choose a random shared secret $a$ with $1 < a < p-1$.  Suppose you send me the corresponding value $A$.  (You don't actually have to send it.)  

In [9]:
g = 65535

a = randint(2, p - 2)
A = pow(g, a, p)

print("a =", a)
print("A =", A)

a = 393023846658356517958096473000390862990670697925458178004953844270364706148237753212314657740786045
A = 1388815479005790788104029519099889470076168378974202076527214289576164042771805022558887932563303870


I am sending you

In [10]:
B = 19472940801104715616883566709623586509945557504365891466465295868554

What is our shared secret?

In [11]:
shared_secret = pow(B, a, p)

shared_secret

1437861812498666176399138664287802660086140307249059321706743660957455935072111788549407929022779635

3. Suppose Alice and Bob are agreeing on a shared secret using Diffie-Hellman and you are intercepting the following values:

In [12]:
g = 65535
p = 65390067647
A = 45011955901
B = 7828173442

Using Shanks's algorithm for the Discrete Logarithm, break their system and determine their shared secret.

In [13]:
a = sbsgs_dl(g, p, A)

shared_secret = pow(B, a, p)

print("a =", a)
print("shared secret =", shared_secret)

a = 65065978829
shared secret = 31014964048


## Problem 3

Suppose Daniel's RSA public key is $(n,e)$ where:

In [14]:
n = 220128835277449925107431266612427384407262202305202431607111
e = 128024737561552881724963970278037549376957617171184078187307

1. Implement Pollard's $p-1$ method in SageMath.  Use your program to factor $n$.

In [15]:
def pollard_p_1_factorization(N):
    a = 2
    for j in range(a, N):
        a = pow(a, j, N)
        g = gcd(a - 1, N)

        if 1 < g < N:
            return g

    return None

In [16]:
p = pollard_p_1_factorization(n)
q = n // p

print("p =", p)
print("q =", q)

p = 4535087767058620428498001
q = 48539046339167474244858411026329111


2. Suppose you intercept the following message intended for Daniel:

In [17]:
c = 84598919372172888688022468367196208044576711805789063943250

Decrypt the message to find the plaintext.  Give your output as an integer.

In [18]:
phi_N = (p - 1) * (q - 1)
d = inverse_mod(e, phi_N)

m = power_mod(c, d, n)

m

99080589798762577571939920254172143270395467297876

3. You speculate the message you intercepted may have been a text message.  Decode the message in the standard way (see hw 5 for instance) to reveal the original text.

In [22]:
def decode(n):
    result = ""

    while n > 0:
        char_code = n % 128
        result += chr(char_code)
        n //= 128

    return result

print(decode(m))

That was a fun semester!


## Problem 4

You are setting up an ElGamal digital signature scheme.  Suppose you choose $g = 65535$ and the prime

In [24]:
p = 73559915962330512031280686662156169308505718695277291442734822801505477158510327

You choose the secret signing key

In [25]:
s = 23150451072664244323965314223703864468298689467773772029888865732054224222678958

1. Compute the verification key $v$.

In [26]:
g = 65535

v = Mod(g, p)^s
v

70722757271320252012945911575963478842533489049204052487914830783497165528629203

2.  Encode the following sentence as an integer $D$ in the usual way using the `encode()` function from hw 5:  "The nopal is nutritious." Give the integer $D$ you obtain.

In [27]:
def encode(a_string):
    return sum([ord(a_string[i])*128^i for i in range(len(a_string))])

D = encode("The nopal is nutritious.")
D

137105315751234996754896597746186908220860676011092

3. Compute the digital signature $(S_{1},S_{2})$ of the document $D$ using the ephemeral key:

In [28]:
e = 128024737561552881724963970278037549376957617171184078187307

In [29]:
S1 = Mod(g, p)^e
S2 = Mod(D - s*int(S1), p - 1) * inverse_mod(e, p - 1)

S1 = int(S1)
S2 = int(S2)

print(S1)
print(S2)

70842882232607255688103968826557726135053981346917544552728716637502429238349129
61354188464848857025931686181627742438683299346040036500479226784351863768203680


4. Verify the signature $(S_{1},S_{2})$ using the verification key $v$.

In [31]:
def verify_signature(m,sign,p,g,k_public,*t):

    p = int(p)
    g = int(g)
    m = int(m)

    decode_message = decode(m)

    q_1 = int(Mod(g,p)^m)
    # print(q_1)
    q_2 = int((Mod(k_public,p)^sign[0])*(Mod(sign[0],p)^sign[1]))
    # print(q_2)

    if q_1 == q_2:
        if len(t) >= 1:
            print(f'The message you sent me is: "{decode_message}".  The signature is also valid.')
        else:
            print('The signature is valid.')
        return True
    else:
        print("Either the message was corrupted or the signature is not valid.  Redo.")
        return False

In [32]:
verify_signature(D, [S1, S2], p, g, v)

The signature is valid.


True